In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondStructure import FixedRateBondStructure, FixedRateBondStructureFunctionMap
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue, FixedRateBondValueFunctionMap

from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

from RVUtils.Interpolation.GeneralCurveInterpolator import GeneralCurveInterpolator
from RVUtils.ust_viz import plot_usts, plot_usts_comparison
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [3]:
usts_mdp = FixedRateBondsMDP(source="USTS_WEBULL_WSJ_LIVE-RL")
usts_tb = FixedRateBondsTB(usts_mdp)

In [49]:
# usts_mdp.get_pricer(request=dict(
#     cusips=["CT10"],
#     timestamp=NY_tz.localize(datetime.datetime(2025, 10, 23, 15, 00)),
# ))

pricer = usts_mdp._get_single_pricer(cusip="CT10", timestamp=NY_tz.localize(datetime.datetime(2025, 10, 23, 15, 00)))
pricer.clean_price()

102.08393327668175

In [13]:
start = NY_tz.localize(datetime.datetime(2025, 10, 22, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 10, 22, 15, 00))
ts_range = pd.date_range(start=start, end=end, freq="1min")

df = usts_tb.get_timeseries(
    start=None, end=None, 
    timestamps=ts_range, 
    queries=[
        # FixedRateBondQuery(cusip="o10/CT10"),
        # FixedRateBondQuery(cusip="oo10/CT10"),
        FixedRateBondQuery(cusip="oo20/CT20"),
    ])
df

PRICING FIXED-RATE BONDS.: 100%|██████████| 481/481 [00:15<00:00, 30.67it/s]


,oo20/CT20 CURVE YTM
Date,
2025-10-22 07:00:00-04:00,0.2
2025-10-22 07:01:00-04:00,0.2
2025-10-22 07:02:00-04:00,0.2
2025-10-22 07:03:00-04:00,0.2
2025-10-22 07:04:00-04:00,0.2
...,...
2025-10-22 14:56:00-04:00,0.2
2025-10-22 14:57:00-04:00,0.2
2025-10-22 14:58:00-04:00,0.2


In [14]:
# plot, fig, ax, ax2, legend = make_secondary_axis_plot(ylabel_left="RATE", ylabel_right="RATE", title=None, engine="plotly")
# plot(
#     df["o10/CT10 CURVE YTM"],
#     which="left",
#     indicators=[
        # {"kind": "last", "show_date": True, "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},
        # {"kind": "sma", "window": 20, "style": {"linestyle": "--", "color": "red"}},
        # {"kind": "simple_avg", "label": "Long-run mean", "style": {"linestyle": "--", "color": "red"}},
        # {"kind": "cum_change", "from_at": "2025-09-24", "label": "Chg from Logan TCGR Speech"},
        # {"kind": "hurst", "hide": True},
        # {"kind": "vol", "returns": "normal", "window": 20, "hide": True},
        # {"kind": "half_life", "method": "ou", "demean": True, "style": {"linestyle": ":"}, "hide": True},
    # ],
    # ou={"enable": True, "steps": 90, "add_metrics_to_legend": True}
# )
# legend(show_date=True)
# plt.show()

plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(df["oo20/CT20 CURVE YTM"], which="left")
legend(valfmt="{:.3f}", show_date=True)